In [0]:
%run ./00_config

#### Passo 1.7 - Criar um relatório simples de nulos

Objetivo: Medir ausência somente nas colunas que realmente serão usadas.
Por que isso importa em crédito: Preencher tudo com zero mistura “não informado”, “não possui” e valor numérico zero.


EXT_SOURCE = score externo normalizado; não sabemos exatamente qual fornecedor ou fórmula gerou cada um.

In [0]:
from pyspark.sql import functions as F

In [0]:
application = spark.table(f"{BRONZE}.application_train")
bureau = spark.table(f"{BRONZE}.bureau")
previous = spark.table(f"{BRONZE}.previous_application")
installments = spark.table(f"{BRONZE}.installments_payments")
credit_card = spark.table(f"{BRONZE}.credit_card_balance")

In [0]:
application_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3" 
                    
]


null_rates = application.agg(*[
    F.avg(F.col(c).isNull().cast("double")).alias(c)
    for c in application_cols
])

display(null_rates)

In [0]:
application.columns

#### Passo 1.8 - Criar a camada Silver

**Objetivo**: Padronizar tipos e criar regras básicas sem ainda agregar históricos.
**Por que isso importa em crédito**: Silver deve ser confiável e reutilizável por vários produtos analíticos.


In [0]:
application_silver = (
    application
    .withColumn(
        "days_employed_anomaly",
        F.when(F.col("DAYS_EMPLOYED") == 365243, 1).otherwise(0)
    )
    .withColumn(
        "DAYS_EMPLOYED_CLEAN",
        F.when(F.col("DAYS_EMPLOYED") == 365243, F.lit(None).cast("double"))
            .otherwise(F.col("DAYS_EMPLOYED").cast("double"))
    )
)

installments_silver = (
    installments
    .withColumn("AMT_INSTALMENT", F.col("AMT_INSTALMENT").cast("double"))
    .withColumn("AMT_PAYMENT", F.col("AMT_PAYMENT").cast("double"))
    .withColumn("DAYS_INSTALMENT", F.col("DAYS_INSTALMENT").cast("double"))
    .withColumn("DAYS_ENTRY_PAYMENT", F.col("DAYS_ENTRY_PAYMENT").cast("double"))
)


In [0]:
installments.columns

In [0]:
installments.schema

In [0]:
credit_card.select("MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF").schema

In [0]:
silver_frames = {
    "application_train": application_silver, 
    "bureau": bureau,
    "previous_application": previous,
    "installments_payments": installments_silver,
    "credit_card_balance": credit_card
}

for table_names, df in silver_frames.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{table_names}")
    )

In [0]:
spark.sql(
    f"DROP TABLE IF EXISTS {SILVER}.installments_paysments"
)